In [ ]:
%cd /home/anw2067/visualnav-transformer/train
import argparse
from datetime import datetime
import os
import torch
import yaml
import copy
import wandb
import json
import random


from nymeria.download_utils import DownloadManager
from nymeria.definitions import DataGroups
from nymeria.data_provider import SequencePathProvider, NymeriaDataProvider
from nymeria.definitions import Subpaths, VrsFiles
from nymeria.recording_data_provider import create_recording_data_provider

import numpy as np
from torchvision import transforms
from dreamsim import dreamsim
from scipy.spatial.transform import Rotation as R
from torch.utils.data import DistributedSampler, RandomSampler, DataLoader
from diffusers.models import AutoencoderKL

from peva.models import CDiT_models
from peva.diffusion import create_diffusion

from vint_train.training.nymeria_training_utils import get_action_smpl_torch
from vint_train.data.misc import XSensConstants, XsensSkeleton
from planning.utils import _compute_pose_and_loss, _compute_part_distance_matrices
from planning.cem import CEMPlanner
from planning.utils import get_nymeria_dataset, load_peva, load_policy
from planning.wrappers import EvaluatorPeva, EvaluatorWaypoint, ObjectiveDreamSIM, PevaWM, Preprocessor, WaypointWM
from planning.sampling import waypoint_sample
from planning.vis_utils import *

from torchvision.utils import save_image

OUTPUT_DIR = "/home/anw2067/visualnav-transformer/train/logs/paper_vis/teaser"
DATA_SAVE_DIR = "/home/anw2067/scratch/nymeria_camera_dir"
DATA_JSON="/home/anw2067/visualnav-transformer/data_jsons/visibility_no_data.json"

os.makedirs(DATA_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
GLOBAL_POLICY = None
GLOBAL_POLICY_DIFFUSION = None
GLOBAL_NOMAD_STATS = None
GLOBAL_NOMAD_CONFIG = None
GLOBAL_PEVA_MODEL = None
GLOBAL_PEVA_DIFFUSION = None
GLOBAL_PEVA_VAE = None
GLOBAL_PEVA_STATS = None
GLOBAL_PEVA_CONFIG = None


In [ ]:

from io import BytesIO
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors

import importlib
import vint_train.visualizing.nymeria_utils
importlib.reload(vint_train.visualizing.nymeria_utils)
from vint_train.visualizing.nymeria_utils import plot_trajs_and_points_full_body



from vint_train.visualizing.nymeria_utils import plot_trajs_and_points_full_body
def custom_plot(body, rotmats=None, points=None, size=1, xlim=None, ylim=None, zlim=None, rpy=None, goal_image=None, ang=0):
    """
    Plot the skeleton of the body
    body: (num_joints, 3) or (num_samples, num_joints, 3)
    rotmats: (num_joints, 3) or (num_samples, num_joints, 3)
    points: (N, 3)
    size: size of the plot
    rpy: (num_joints, 3) or (num_samples, num_joints, 3) - Roll, Pitch, Yaw angles in radians
    goal_image: (3, H, W) or (H, W, 3) - Goal image to plot perpendicular to head x-axis
    
    Returns:
    pil_img: PIL image
    """
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Remove entire grid - no grid lines, no axis lines, no ticks, no labels
    ax.grid(False)
    # Remove all panes (background planes)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('white')
    ax.yaxis.pane.set_edgecolor('white')
    ax.zaxis.pane.set_edgecolor('white')
    # Remove tick labels
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    # Remove ticks
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    # INSERT_YOUR_CODE
    ax.set_axis_off()
    
    if body.ndim == 2:
        body = body[None]
        if rotmats is not None:
            rotmats = rotmats[None]
        if rpy is not None:
            rpy = rpy[None]
    
    N = body.shape[0]
    
    if N == 1:
        colors = [XSensConstants.color_skeleton[:XSensConstants.upper_body_num_parts]]
    elif N > 1:
        # one color for each body
        # Use matplotlib's color cycle to assign one color per body for N > 1
        color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
        colors = [
            255. * np.tile(
                np.array(mcolors.to_rgb(color_cycle[i % len(color_cycle)])),
                (XSensConstants.upper_body_num_parts, 1)
            ) for i in range(N)
        ]
        
    if points is not None:
        points = np.asarray(points)
        if points.ndim == 1:
            points = points[None, :]
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], c='r', s=30, marker='*')
    
    # Plot head orientation using RPY if available
    if rpy is not None:
        from scipy.spatial.transform import Rotation as R
        # Convert to numpy if torch tensor
        rpy_np = rpy.detach().cpu().numpy() if isinstance(rpy, torch.Tensor) else np.asarray(rpy)
        body_np = body.detach().cpu().numpy() if isinstance(body, torch.Tensor) else np.asarray(body)
        
        head_idx = XSensConstants.part_names.index("Head")
        for i in range(N):
            head_rpy = rpy_np[i, head_idx, :]  # (3,) - Roll, Pitch, Yaw in radians
            head_pos = body_np[i, head_idx, :]  # (3,) - Head position
            
            # Convert RPY to rotation matrix
            # RPY is in 'xyz' order (roll around x, pitch around y, yaw around z)
            rot = R.from_euler('xyz', head_rpy, degrees=False)
            rot_matrix = rot.as_matrix()  # (3, 3)
            
            # Plot orientation arrows: X (red), Y (green), Z (blue)
            # Use a reasonable arrow length based on the skeleton size
            # arrow_length = 0.15
            # ax.quiver(head_pos[0], head_pos[1], head_pos[2], 
            #          rot_matrix[0, 0], rot_matrix[1, 0], rot_matrix[2, 0], 
            #          color='r', length=arrow_length, normalize=True, arrow_length_ratio=0.3)  # +X Forward (Red)
            # ax.quiver(head_pos[0], head_pos[1], head_pos[2], 
            #          rot_matrix[0, 1], rot_matrix[1, 1], rot_matrix[2, 1], 
            #          color='g', length=arrow_length, normalize=True, arrow_length_ratio=0.3)  # +Y Left (Green)
            # ax.quiver(head_pos[0], head_pos[1], head_pos[2], 
            #          rot_matrix[0, 2], rot_matrix[1, 2], rot_matrix[2, 2], 
            #          color='b', length=arrow_length, normalize=True, arrow_length_ratio=0.3)  # +Z Up (Blue)
            
            # Plot goal image perpendicular to head x-axis if provided
            if goal_image is not None and i == 0:  # Only plot for the first body
                # Convert goal_image to numpy if needed
                if isinstance(goal_image, torch.Tensor):
                    goal_img_np = goal_image.detach().cpu().numpy()
                else:
                    goal_img_np = np.asarray(goal_image)
                
                # Handle different image formats: (3, H, W) or (H, W, 3)
                if goal_img_np.shape[0] == 3:
                    goal_img_np = np.transpose(goal_img_np, (1, 2, 0))
                
                # Normalize to [0, 1] if needed
                if goal_img_np.max() > 1.0:
                    goal_img_np = goal_img_np / 255.0
                goal_img_np = np.clip(goal_img_np, 0, 1)
                
                # Flip image vertically to correct orientation
                goal_img_np = np.flipud(goal_img_np)
                
                # Extract yaw from head RPY and add 45 degrees
                # RPY is in 'xyz' order: [roll, pitch, yaw]
                head_yaw = head_rpy[2]  # Yaw is the third element
                image_yaw = head_yaw + np.deg2rad(45)  # Add 45 degrees
                
                # Create rotation matrix with roll=0, pitch=0, yaw=image_yaw
                from scipy.spatial.transform import Rotation as R
                image_rot = R.from_euler('xyz', [0, 0, image_yaw], degrees=False)
                image_rot_matrix = image_rot.as_matrix()  # (3, 3)
                
                # Get axes from the image rotation matrix
                # X-axis is forward (perpendicular to image plane)
                image_x_axis = image_rot_matrix[:, 0]  # Forward direction
                image_y_axis = image_rot_matrix[:, 1]  # Left/right direction
                image_z_axis = image_rot_matrix[:, 2]  # Up direction
                
                # Image dimensions
                img_h, img_w = goal_img_np.shape[:2]
                img_size = 0.3  # Size of the image plane in 3D space
                
                # Create a plane perpendicular to image x-axis
                # The plane is defined by y and z axes
                u = np.linspace(-img_size/2, img_size/2, img_w)
                v = np.linspace(-img_size/2, img_size/2, img_h)
                U, V = np.meshgrid(u, v)
                
                # Position the plane at head position, offset along image x-axis
                offset_distance = 0.2  # Distance from head along x-axis
                plane_center = head_pos + image_x_axis * offset_distance
                
                # Create plane coordinates in 3D
                X = plane_center[0] + U * image_y_axis[0] + V * image_z_axis[0]
                Y = plane_center[1] + U * image_y_axis[1] + V * image_z_axis[1]
                Z = plane_center[2] + U * image_y_axis[2] + V * image_z_axis[2]
                
                # Plot the image as a surface
                ax.plot_surface(X, Y, Z, facecolors=goal_img_np, rstride=1, cstride=1, shade=False)
    
    # Draw lines from specific joints of second skeleton to head of first skeleton
    if N >= 2:
        # Convert to numpy if needed
        body_np = body.detach().cpu().numpy() if isinstance(body, torch.Tensor) else np.asarray(body)
        
        # Get head position of first skeleton (index 0, joint 6 is Head)
        head_idx = XSensConstants.part_names.index("Head")
        first_head_pos = body_np[0, head_idx, :]  # (3,)
        
        # Joint indices to draw lines from: [0 (Pelvis), 6 (Head), 10 (R_Hand), 14 (L_Hand)]
        joint_indices = [0, 6, 10, 14]
        
        # Get positions of these joints from second skeleton (index 1)
        for joint_idx in joint_indices:
            second_joint_pos = body_np[1, joint_idx, :]  # (3,)
            
            # Draw thin black line
            ax.plot([second_joint_pos[0], first_head_pos[0]], 
                   [second_joint_pos[1], first_head_pos[1]], 
                   [second_joint_pos[2], first_head_pos[2]], 
                   'k-', linewidth=1, alpha=0.7)
    
    # Compute bounds from all skeleton data to accommodate all bodies (after rotation)
    if N > 0:
        # Get all joint positions from all bodies
        all_positions = body.reshape(-1, 3)  # Flatten to (N*num_joints, 3)
        
        # Add points if they exist
        if points is not None:
            all_positions = np.vstack([all_positions, points])
        
        # Compute centers and ranges for each axis
        x_center = (all_positions[:, 0].min() + all_positions[:, 0].max()) / 2
        y_center = (all_positions[:, 1].min() + all_positions[:, 1].max()) / 2
        z_center = (all_positions[:, 2].min() + all_positions[:, 2].max()) / 2
        
        x_range = all_positions[:, 0].max() - all_positions[:, 0].min()
        y_range = all_positions[:, 1].max() - all_positions[:, 1].min()
        z_range = all_positions[:, 2].max() - all_positions[:, 2].min()
        
        # Use the maximum range for x and y to maintain square aspect ratio
        # Increase xy range to give more space between skeletons, reduce z to prevent stretching
        max_xy_range = max(x_range, y_range, 0.1)  # Ensure minimum range
        # Increase xy range to create more distance between skeletons
        scaled_xy_range = max_xy_range * 1.2
        padding_xy = max(scaled_xy_range * 0.02, 0.02)  # Minimal padding
        
        # Scale down z range aggressively to reduce stretching (shortened by 35%)
        scaled_z_range = z_range * 0.325 if z_range > 0 else 0.1625
        padding_z = max(scaled_z_range * 0.05, 0.05)  # Minimal padding
        
        # Set symmetric limits around the center with scaled ranges
        if xlim is None:
            ax.set_xlim(x_center - scaled_xy_range/2 - padding_xy, x_center + scaled_xy_range/2 + padding_xy)
        if ylim is None:
            ax.set_ylim(y_center - scaled_xy_range/2 - padding_xy, y_center + scaled_xy_range/2 + padding_xy)
        if zlim is None:
            ax.set_zlim(z_center - scaled_z_range/2 - padding_z, z_center + scaled_z_range/2 + padding_z)
    
    # Rotate camera yaw to match the first skeleton's head yaw
    if N > 0 and rpy is not None:
        # Convert to numpy if needed
        rpy_np = rpy.detach().cpu().numpy() if isinstance(rpy, torch.Tensor) else np.asarray(rpy)
        
        # Get first skeleton's head RPY
        head_idx = XSensConstants.part_names.index("Head")
        first_head_rpy = rpy_np[0, head_idx, :]  # (3,) - Roll, Pitch, Yaw in radians
        
        # Extract yaw (third element in 'xyz' order)
        head_yaw = first_head_rpy[2]
        
        # Convert yaw to azimuth angle (in degrees)
        # Yaw rotation around z-axis corresponds to azimuth
        azim = np.rad2deg(head_yaw)
        
        # Set elevation to look perpendicular to the displacement (if multiple skeletons)
        if N > 1:
            # Extract pelvis locations (index 0) from each body
            body_np = body.detach().cpu().numpy() if isinstance(body, torch.Tensor) else np.asarray(body)
            pelvis_locations = body_np[:, 0, :]  # (N, 3)
            
            # Find the pair with maximum distance
            max_dist = 0
            max_pair = (0, 1)
            for i in range(N):
                for j in range(i + 1, N):
                    dist = np.linalg.norm(pelvis_locations[i] - pelvis_locations[j])
                    if dist > max_dist:
                        max_dist = dist
                        max_pair = (i, j)
            
            # Calculate direction vector between the two pelvis locations
            direction = pelvis_locations[max_pair[1]] - pelvis_locations[max_pair[0]]
            direction_norm = np.linalg.norm(direction)
            
            if direction_norm > 1e-6:  # Avoid division by zero
                direction = direction / direction_norm
                
                # Find a vector perpendicular to the displacement direction
                z_axis = np.array([0, 0, 1])
                perpendicular = np.cross(direction, z_axis)
                
                # If perpendicular is too small, use x-axis instead
                if np.linalg.norm(perpendicular) < 1e-6:
                    x_axis = np.array([1, 0, 0])
                    perpendicular = np.cross(direction, x_axis)
                
                # Normalize and get elevation
                perp_norm = np.linalg.norm(perpendicular)
                if perp_norm > 1e-6:
                    perpendicular = perpendicular / perp_norm
                    elev = np.arcsin(perpendicular[2]) * 180 / np.pi
                else:
                    elev = 0
            else:
                elev = 0
        else:
            elev = 0
        
        # Add rotation offset if specified
        azim = azim + ang
        
        # Set the view with yaw matching first skeleton's head
        ax.view_init(elev=elev, azim=azim)
    
    for i in range(N):
        plot_trajs_and_points_full_body(
            ax,
            body[i],
            rotmats[i] if rotmats is not None else None,
            XSensConstants.kintree_parents[:XSensConstants.upper_body_num_parts],
            colors[i],
            size=size,
        )
        
    
    
    # Apply user-specified limits if provided (these override computed limits)
    if xlim is not None:
        ax.set_xlim(xlim[0], xlim[1])
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])
    if zlim is not None:
        ax.set_zlim(zlim[0], zlim[1])
    
    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0., dpi=300)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    return pil_img

In [ ]:
import importlib
import planning.vis_utils
from vint_train.visualizing.nymeria_utils import plot_skeleton
importlib.reload(planning.vis_utils)
from planning.vis_utils import *
import vint_train.data.misc
importlib.reload(vint_train.data.misc)
from vint_train.data.misc import XSensConstants, XsensSkeleton 
from pathlib import Path
disable_logging()

def main(args):
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    global GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG
    global GLOBAL_PEVA_MODEL, GLOBAL_PEVA_DIFFUSION, GLOBAL_PEVA_VAE, GLOBAL_PEVA_STATS, GLOBAL_PEVA_CONFIG
    if GLOBAL_POLICY is None:
        GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG = load_policy(args.nomad_config, args.nomad_checkpoint, device=device)
        # GLOBAL_PEVA_MODEL, _, GLOBAL_PEVA_DIFFUSION, GLOBAL_PEVA_VAE, GLOBAL_PEVA_STATS, GLOBAL_PEVA_CONFIG = load_peva(args.peva_config, args.peva_checkpoint, device=device,
        #                                                             inference_context_size=args.peva_context_size,
        #                                                             diffusion_steps=args.peva_diffusion_steps)
    policy, policy_diffusion, nomad_stats, nomad_config = GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG
    # peva_model, _, peva_diffusion, peva_vae, peva_stats, peva_config = GLOBAL_PEVA_MODEL, None, GLOBAL_PEVA_DIFFUSION, GLOBAL_PEVA_VAE, GLOBAL_PEVA_STATS, GLOBAL_PEVA_CONFIG
    # policy, policy_diffusion, nomad_stats, nomad_config = load_policy(args.nomad_config, args.nomad_checkpoint, device=device)
    # peva_model, _, peva_diffusion, peva_vae, peva_stats, peva_config = load_peva(args.peva_config, args.peva_checkpoint, device=device,
                                                                    # inference_context_size=args.peva_context_size,
                                                                    # diffusion_steps=args.peva_diffusion_steps)
    
    dataset = get_nymeria_dataset(nomad_config, context_size=max(args.peva_context_size-1, nomad_config["context_size"]), goal_timestep_offset=args.goal_timestep_offset)
    sampler = DistributedSampler(dataset, num_replicas=1, rank=0, shuffle=args.shuffle, seed=seed)
    dataloader = DataLoader(dataset, batch_size=1, sampler=sampler, num_workers=1)
    
    count = 0
    prev_track_name = None
    for idx, batch in enumerate(dataloader):
        obs_images = batch["obs_images"] # 1, context_size, 3, H, W
        goal_image = batch["goal_image"] # 1, 3, H, W
        context_poses = batch["context_poses"] # 1, context_size, 48

        deltas = batch["deltas"] # 1, horizon, action_dim
        first_pose = batch["first_pose"] # 1, 1, 48
        xsens_offsets = batch["xsens_offsets"][0] # 1, 15, 3
        goal_obs = batch["goal_obs"] # 1, 3, H, W
        goal_image_coords = batch["goal_image_coords"] # 1, 23, 2
        
        dataset_index = batch["dataset_index"].item()
        track_name, track_index = batch["dataset_track"][0], batch["dataset_track_index"].item()
        track_idx_name = f"{track_name}-{track_index}"
        
        skel = XsensSkeleton(xsens_offsets)
        gt_actions = get_action_smpl_torch(first_pose, deltas, XSensConstants.upper_body_num_parts) # B, T, 48
        xyz_dist_matrix, _, init_xyz, _ = _compute_part_distance_matrices(first_pose[:, -1], gt_actions[:, -1], skel)
        visible_plus_head = (goal_image_coords != -1).all(dim=-1)[:, :XSensConstants.upper_body_num_parts] # B, num_parts
        visible_plus_head[:, XSensConstants.part_names.index("Head")] = True
        init_visible_plus_head = xyz_dist_matrix[:, XSensConstants.leaf_indices] * visible_plus_head[:, XSensConstants.leaf_indices]
        init_visible_plus_head = (init_visible_plus_head.sum() / visible_plus_head.sum()).item()
        
        if not args.keep_nonvisible_goal:
            visible = False
            find_count = 0
            for part in ["Pelvis", "Head", "R_Hand", "L_Hand"]:
                index = XSensConstants.part_names.index(part)
                if all(goal_image_coords[0, index] != -1):
                    find_count += 1
                    if find_count == 4:
                        visible = True
                        break
            if not visible:
                print(f"No visible parts in {track_idx_name}")
                continue
            
        if init_visible_plus_head < args.min_dist_threshold:
            print(f"Initial distance of visible + head joints is less than {args.min_dist_threshold} in {track_idx_name}")
            continue
        print(f"Visualizing {track_idx_name}")
        
        curr_save_dir = f"{OUTPUT_DIR}/{track_idx_name}"
        os.makedirs(curr_save_dir, exist_ok=True)
        save_img = torch.cat([obs_images, goal_obs[None],torch.zeros_like(obs_images[:, :-2]), goal_image[None]], dim=1)[0]
        save_image(save_img, f"{curr_save_dir}/context_and_goal.png", nrow=obs_images.shape[1])
        
        # config details
        policy_pred_horizon = nomad_config["len_traj_pred"]
        policy_action_dim = nomad_config["input_dims"]
        image_size = nomad_config["image_size"][0]
        policy_context_size = nomad_config["context_size"] + 1
        # peva_context_size = peva_config["context_size"]
        peva_latent_size = image_size // 8
        
        waypoints = goal_image_coords[:, XSensConstants.leaf_indices].flatten(1, 2)[:, None] # 1, 4, 2
        policy_context_poses = context_poses[:, -policy_context_size:]
        
        gt_actions = get_action_smpl_torch(first_pose, deltas, XSensConstants.upper_body_num_parts) # B, T, 48
        fp_xyz, fp_rpy = forward_kinematics_wrapper(first_pose, skel, return_euler=True) # (B, 1, 15, 3), (B, 1, 15, 3)
        gt_xyz, gt_rpy = forward_kinematics_wrapper(gt_actions[:, -1:], skel, return_euler=True) # (B, 1, 15, 3), (B, 1, 15, 3)

        pelvis_l2_dist = torch.norm(fp_xyz[0, 0, 0] - gt_xyz[0, 0, 0], dim=-1)
        
        goal_image_np = goal_image.cpu().numpy()[0]  # (3, H, W)
        
        # if track_idx_name != "20231110_s0_thomas_brown_act3_pisdac-1387":
        #     continue
        
        img_dir = "/home/anw2067/visualnav-transformer/train/logs/paper_vis/waypoint_generation_vis/pelvis_distance"
        os.makedirs(img_dir, exist_ok=True)
        # os.makedirs(save_dir, exist_ok=True)
        # for ang in range(-45, 45, 15):
        img = custom_plot(torch.cat([fp_xyz[0], gt_xyz[0]], dim=0), 
                        rpy=torch.cat([fp_rpy[0], gt_rpy[0]], dim=0),
                        goal_image=goal_image_np)
        # img.show()
        # break
            
        img.save(os.path.join(img_dir, f"{track_idx_name}-dist{pelvis_l2_dist.item():.3f}.png"))
            
        count += 1
        if count > 30:
            break
        
        
MODEL_DIRECTORY={
    "draw": (   
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/config.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/ema_9.pth"
    ),
    "gravity": (
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_18_11_47:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-preserveUpDown/config.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_18_11_47:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-preserveUpDown/ema_9.pth"
    ),
    "draw_mask": (
        "/home/anw2067/visualnav-transformer/train/config/torch/minimal-nomad-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-waypointMask.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2026_01_21_06_54:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-waypointMask/ema_9.pth"
    )
}
        
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    
    parser.add_argument("-a", "--algo", type=str, choices=["peva", "waypoint"], default="waypoint", help="Planning algorithm")
    parser.add_argument("--use_leafxyz_as_cost", action='store_true', help="Uses the metric(leaf-xyz) instead of a normal cost_fn")
    parser.add_argument("--goal_timestep_offset", type=int, default=None, help="Goal timestep offset")
    parser.add_argument("--shuffle", action="store_true", help="Shuffle the dataset")
    
    # # CEM parameters
    # parser.add_argument("-n", "--num_samples", type=int, default=32, help="Number of samples")
    # parser.add_argument("-t", "--topk", type=int, default=4, help="Top k samples")
    # parser.add_argument("-v", "--var_scale", type=float, default=0.5, help="Variance scale")
    # parser.add_argument("-o", "--opt_steps", type=int, default=8, help="Optimization steps")
    # parser.add_argument("-e", "--eval_every", type=int, default=1, help="Evaluation frequency")
    # parser.add_argument("-H", "--horizon", type=int, default=1, help="Time horizon")
    # parser.add_argument("-N", "--num_eval_samples", type=int, default=1, help="Number of evaluation samples")
    
    parser.add_argument("--keep_nonvisible_goal", action="store_true", help="Keep non-visible goal in the dataset")
    parser.add_argument("--min_index_goal", type=int, default=0, help="Minimum index of the goal to plan")
    parser.add_argument("--min_dist_threshold", type=float, default=0.1, help="Minimum distance threshold")
    parser.add_argument("--num_samples_to_plan", type=int, default=128, help="Number of samples to plan")
    parser.add_argument("--no_wandb", action="store_true", help="Don't use wandb")
    parser.add_argument("--test", action="store_true", help="Test run")
    
    parser.add_argument("--peva_config", type=str, default="/home/anw2067/visualnav-transformer/train/peva/config/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_-64to_64_1_goal_emb_relative_xxl.yaml")
    parser.add_argument("--peva_checkpoint", type=str, default="/scratch/anw2067/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_cancel_scaler_-64to_64_xxl_280_0180000.pth.tar")
    parser.add_argument("--peva_context_size", type=int, default=15, help="PEVA context size")
    parser.add_argument("--peva_diffusion_steps", type=int, default=250, help="PEVA diffusion steps")
    
    parser.add_argument("--nomad_model", type=str, default="draw", choices=["draw", "gravity", "draw_mask"])
    parser.add_argument("--nomad_config", type=str, default=None)
    parser.add_argument("--nomad_checkpoint", type=str, default=None)
    
    parser.add_argument("--world_size", type=int, default=1, help="World size")
    parser.add_argument("--rank", type=int, default=0, help="Rank")
    
    # In Jupyter notebooks, pass arguments as a list to parse_args() instead of using sys.argv
    # Pass an empty list [] to use all defaults, or specify arguments like: ['--shuffle', '--nomad_model', 'draw']
    args = parser.parse_args(["--shuffle", "--nomad_model", "draw_mask", "--peva_context_size", "7"])
    
    if args.nomad_model is not None:
        assert args.nomad_config is None and args.nomad_checkpoint is None
        args.nomad_config, args.nomad_checkpoint = MODEL_DIRECTORY[args.nomad_model]
    else:
        assert args.nomad_config is not None and args.nomad_checkpoint is not None
    
    main(args)